# 03 Geneformer-V2-104M 비교

무엇을 하는가: scGPT와 같은 세포·환자 split에서 V2 공식 TranscriptomeTokenizer와 frozen CLS embedding을 실행한다. 316M root checkpoint나 V1 tokenizer를 섞지 않는다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
assert (ROOT / 'third_party/Geneformer/Geneformer-V2-104M/model.safetensors').exists()
run_script('04_geneformer_annotation.py', 'tutorial_geneformer_smoke.yaml', model='geneformer')

/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-geneformer/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/04_geneformer_annotation.py --config /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_geneformer_smoke.yaml


Tokenizing /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver/geneformer_inputs/tutorial_geneformer_smoke/counts.h5ad
/mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver/geneformer_inputs/tutorial_geneformer_smoke/counts.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
{"run_id": "tutorial_geneformer_smoke", "macro_f1": 0.6944444444444443, "balanced_accuracy": 0.75, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.0}
{'run_id': 'tutorial_geneformer_smoke', 'status': 'executed', 'task': 'annotation', 'dataset_accession': 'GSE149614', 'model_name': 'Geneformer-V2-104M frozen CLS + supervised logistic head', 'seed': 42, 'n_cells': 64, 'data_hash': 'cbbe84169b7362ac0dc19ab51e6af472d7f7f945203ae45d81de5dd0e33693df', 'split_manifest_

CompletedProcess(args=['/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-geneformer/bin/python', '-u', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/04_geneformer_annotation.py', '--config', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_geneformer_smoke.yaml'], returncode=0, stdout='Tokenizing /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver/geneformer_inputs/tutorial_geneformer_smoke/counts.h5ad\n/mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver/geneformer_inputs/tutorial_geneformer_smoke/counts.h5ad has no column attribute \'filter_pass\'; tokenizing all cells.\nCreating dataset.\n{"run_id": "tutorial_geneformer_smoke", "macro_f1": 0.6944444444444443, "balanced_accuracy": 0.75, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shu

In [3]:
audit = json.loads((DATA / 'outputs/tutorial_geneformer_smoke/tokenization_audit.json').read_text())
print(json.dumps(audit, indent=2))
r = json.loads((ROOT / 'results/tutorial_geneformer_smoke/run.json').read_text())
print('actual parameters=', r['actual_parameter_count'])
print('macro-F1=',r['metrics']['macro_f1'], 'peak allocated GiB=',r.get('peak_cuda_allocated_gib'))
print(pd.read_csv(ROOT / 'results/tutorial_geneformer_smoke/per_class.tsv', sep='\t').to_string(index=False))

{
  "n_original_genes": 25712,
  "n_mapped_genes": 18291,
  "n_original_vocab_genes": 16721,
  "n_counts_basis": "original all-gene raw library sum, before mapping/filtering",
  "total_original_counts": 501158,
  "total_mapped_counts": 483262,
  "mapped_count_fraction": 0.9642907027324716,
  "token_length_quantiles": [
    520.0,
    844.0,
    1247.5,
    2203.0,
    4096.0
  ],
  "model_input_size": 4096,
  "special_token": true,
  "collapse_gene_ids": true,
  "duplicate_mapped_ids": 0
}
actual parameters= 104386867
macro-F1= 0.6944444444444443 peak allocated GiB= 0.641481876373291
  Unnamed: 0  precision  recall  f1-score  support
           B   0.500000    0.50  0.500000     2.00
 Endothelial   1.000000    1.00  1.000000     2.00
  Fibroblast   1.000000    1.00  1.000000     2.00
  Hepatocyte   0.500000    1.00  0.666667     2.00
     Myeloid   1.000000    1.00  1.000000     2.00
        T/NK   0.000000    0.00  0.000000     2.00
    accuracy   0.750000    0.75  0.750000     0.75
 

생물학적 해석: raw all-gene count에서 mapping 가능한 유전자를 공식 V2 rank encoding으로 변환한다. n_counts는 gene mapping 이전 library size를 유지한다. 4096 길이에 CLS/EOS가 포함된다. 공식 기본 emb_layer=-1과 같은 hidden_states[11] CLS를 사용한다. 입력 범위가 다른 실용 pipeline 비교다. 사전학습 corpus 중복은 unknown이다.

실패 해결: scGPT의 HVG/binned 데이터를 재사용하지 않는다. Geneformer 전용 환경은 envs/activate.sh geneformer. OOM이면 batch1부터 실행하고 sequence 변경을 별도 기록한다.